In [1]:
"""
═══════════════════════════════════════════════════════════════════
FL Benchmarking — BASE (shared across all model notebooks)
Dataset  : ULB Credit Card Fraud (creditcard.csv)
Contains : Config, HP, data loading, all model classes, helpers,
           privacy mechanisms, local training, metrics, checkpointing
═══════════════════════════════════════════════════════════════════
"""

# ── HYPERPARAMETERS ────────────────────────────────────────────────
HP = {
    "N_BANKS"        : 5,
    "FL_ROUNDS"      : 60,
    "LOCAL_EPOCHS"   : 3,
    "FINETUNE_EPOCHS": 5,
    "LOCAL_STEPS"    : 25,
    "BATCH_SIZE"     : 128,
    "LR"             : 0.001,
    "FINETUNE_LR"    : 0.0003,
    "FEDPROX_MU"     : 0.01,
    "FEDNOVA_RHO"    : 0.9,
    "DIRICHLET_ALPHA": 0.4,
    "DP_CLIP_NORM"   : 1.0,
    "DP_NOISE_MULT"  : 1.1,
    "DP_EPSILON"     : 4.0,
    "TOPK_RATIO"     : 0.01,
    "RANDOM_STATE"   : 42,
    "SAVE_CKPT_EVERY": 10,
    "PRINT_EVERY"    : 1,
}

import os, json, time, copy, warnings, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, roc_curve,
                             f1_score, precision_score, recall_score)
from imblearn.over_sampling import SMOTE
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
warnings.filterwarnings("ignore")

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_ROOT = "outputs"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

ALL_FL      = ["FedAvg", "FedProx", "SCAFFOLD", "FedNova", "PersFL"]
ALL_PRIVACY = ["NoDP", "DP", "Sparsification"]

if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Device : {DEVICE}")
print(f"Output : {OUTPUT_ROOT}/")


# ══════════════════════════════════════════════════════════════════
#  DATA LOADING & SPLITTING
# ══════════════════════════════════════════════════════════════════

def load_data(path="/kaggle/input/datasets/organizations/mlg-ulb/creditcardfraud/creditcard.csv"):
    print("\nLoading dataset...")
    df  = pd.read_csv(path)
    X   = df.drop("Class", axis=1).values.astype(np.float32)
    y   = df["Class"].values
    sc  = StandardScaler()
    X   = sc.fit_transform(X)
    print(f"  Total records : {len(df):,}")
    print(f"  Fraud rate    : {y.mean()*100:.2f}%")
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=HP["RANDOM_STATE"]
    )
    return X_tr, X_te, y_tr, y_te


def non_iid_split(X, y, n=5, seed=42):
    rng        = np.random.default_rng(seed)
    fi         = np.where(y == 1)[0]
    li         = np.where(y == 0)[0]
    splits     = np.round(
        rng.dirichlet(np.ones(n) * HP["DIRICHLET_ALPHA"]) * len(fi)
    ).astype(int)
    splits[-1] = len(fi) - splits[:-1].sum()
    banks, f0  = [], 0
    lpp        = len(li) // n
    for i in range(n):
        fe  = f0 + splits[i]
        idx = np.concatenate([fi[f0:fe], li[i*lpp:(i+1)*lpp]])
        rng.shuffle(idx)
        banks.append((X[idx], y[idx]))
        f0 = fe
    print(f"\nNon-IID split (Dirichlet α={HP['DIRICHLET_ALPHA']}):")
    for i, (xb, yb) in enumerate(banks):
        print(f"  Bank {i+1}: {len(xb):>6,} records | fraud {yb.mean()*100:.2f}%")
    return banks


def smote_bank(X, y):
    """SMOTE capped at 10k rows to prevent memory explosion."""
    try:
        k        = min(3, int(y.sum()) - 1)
        sm       = SMOTE(random_state=HP["RANDOM_STATE"], k_neighbors=k)
        X_r, y_r = sm.fit_resample(X, y)
        if len(X_r) > 10000:
            idx      = np.random.choice(len(X_r), 10000, replace=False)
            X_r, y_r = X_r[idx], y_r[idx]
        return X_r, y_r
    except Exception:
        return X, y


def make_loader(X, y):
    return DataLoader(
        TensorDataset(torch.tensor(X, dtype=torch.float32),
                      torch.tensor(y, dtype=torch.float32)),
        batch_size=HP["BATCH_SIZE"], shuffle=True
    )


# ══════════════════════════════════════════════════════════════════
#  ML MODELS
# ══════════════════════════════════════════════════════════════════

class LRWrapper:
    def __init__(self, n_features):
        self.model = LogisticRegression(
            max_iter=1000, random_state=HP["RANDOM_STATE"],
            class_weight="balanced", solver="lbfgs", warm_start=True
        )
        self._fitted    = False
        self.n_features = n_features

    def fit(self, X, y):
        self.model.fit(X, y)
        self._fitted = True

    def predict_proba(self, X):
        if not self._fitted:
            return np.full(len(X), 0.5)
        return self.model.predict_proba(X)[:, 1]

    def get_params(self):
        if not self._fitted:
            return {"coef": np.zeros((1, self.n_features)), "intercept": np.zeros(1)}
        return {"coef": self.model.coef_.copy(), "intercept": self.model.intercept_.copy()}

    def set_params(self, params):
        if not self._fitted:
            X_d = np.random.randn(10, self.n_features)
            y_d = np.array([0]*9 + [1])
            self.model.fit(X_d, y_d)
            self._fitted = True
        self.model.coef_      = params["coef"].copy()
        self.model.intercept_ = params["intercept"].copy()

    def copy(self):
        new = LRWrapper(self.n_features)
        if self._fitted:
            new.set_params(self.get_params())
            new._fitted = True
        return new


class MLP(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32),  nn.ReLU(),
            nn.Linear(32, 1),   nn.Sigmoid()
        )
    def forward(self, x): return self.net(x).squeeze()


class TabNet(nn.Module):
    def __init__(self, in_dim, n_steps=3, n_dim=64):
        super().__init__()
        self.n_steps   = n_steps
        self.step_attn = nn.ModuleList([
            nn.Sequential(nn.Linear(in_dim, in_dim), nn.Softmax(dim=1))
            for _ in range(n_steps)
        ])
        self.step_fc = nn.ModuleList([
            nn.Sequential(
                nn.Linear(in_dim, n_dim), nn.LayerNorm(n_dim), nn.ReLU(),
                nn.Linear(n_dim, n_dim),  nn.LayerNorm(n_dim), nn.ReLU()
            )
            for _ in range(n_steps)
        ])
        self.final = nn.Sequential(nn.Linear(n_dim, 1), nn.Sigmoid())

    def forward(self, x):
        h = torch.zeros(x.size(0), self.step_fc[0][-2].normalized_shape[0], device=x.device)
        for i in range(self.n_steps):
            h = h + self.step_fc[i](x * self.step_attn[i](x))
        return self.final(h / self.n_steps).squeeze()


class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim*2), nn.LayerNorm(dim*2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(dim*2, dim), nn.LayerNorm(dim)
        )
        self.act = nn.GELU()
    def forward(self, x): return self.act(x + self.block(x))


class ResNet(nn.Module):
    def __init__(self, in_dim, hidden=64, n_blocks=2):
        super().__init__()
        self.proj   = nn.Sequential(nn.Linear(in_dim, hidden), nn.LayerNorm(hidden), nn.GELU())
        self.blocks = nn.Sequential(*[ResBlock(hidden) for _ in range(n_blocks)])
        self.head   = nn.Sequential(nn.Linear(hidden, 32), nn.GELU(),
                                    nn.Dropout(0.2), nn.Linear(32, 1), nn.Sigmoid())
    def forward(self, x): return self.head(self.blocks(self.proj(x))).squeeze()


def build_model(model_name, n_features):
    if model_name == "LR":
        return LRWrapper(n_features)
    elif model_name == "MLP":
        return MLP(n_features).to(DEVICE)
    elif model_name == "TabNet":
        return TabNet(n_features).to(DEVICE)
    elif model_name == "ResNet":
        return ResNet(n_features).to(DEVICE)
    else:
        raise ValueError(f"Unknown model: {model_name}")


# ══════════════════════════════════════════════════════════════════
#  PRIVACY MECHANISMS
# ══════════════════════════════════════════════════════════════════

def apply_privacy(model, privacy_mode, n_samples=1):
    if privacy_mode == "NoDP":
        return
    elif privacy_mode == "DP":
        total_norm = sum(
            p.grad.data.norm(2).item() ** 2
            for p in model.parameters() if p.grad is not None
        ) ** 0.5
        clip_coef  = HP["DP_CLIP_NORM"] / max(total_norm, HP["DP_CLIP_NORM"])
        noise_std  = HP["DP_NOISE_MULT"] * HP["DP_CLIP_NORM"] / max(n_samples, 1)
        for p in model.parameters():
            if p.grad is not None:
                p.grad.data.mul_(clip_coef)
                p.grad.data.add_(torch.randn_like(p.grad.data) * noise_std)
    elif privacy_mode == "Sparsification":
        for p in model.parameters():
            if p.grad is not None:
                grad_flat = p.grad.data.abs().flatten()
                k         = max(1, int(len(grad_flat) * HP["TOPK_RATIO"]))
                threshold = torch.topk(grad_flat, k).values[-1]
                mask      = p.grad.data.abs() >= threshold
                p.grad.data.mul_(mask.float())


# ══════════════════════════════════════════════════════════════════
#  LOCAL TRAINING
# ══════════════════════════════════════════════════════════════════

def local_train_nn(model, loader, epochs, lr, privacy_mode,
                   global_model=None, mu=0.0,
                   c_local=None, c_global=None,
                   use_steps=False, n_steps=25):
    model.train()
    optimizer  = optim.Adam(model.parameters(), lr=lr)
    criterion  = nn.BCELoss()
    dev        = next(model.parameters()).device
    aux        = {}

    if use_steps:
        global_state = copy.deepcopy(model.state_dict())
        data_iter    = iter(loader)
        for step in range(n_steps):
            try:
                X_b, y_b = next(data_iter)
            except StopIteration:
                data_iter = iter(loader)
                X_b, y_b = next(data_iter)
            X_b, y_b = X_b.to(dev), y_b.to(dev)
            optimizer.zero_grad()
            loss = criterion(model(X_b), y_b)
            loss.backward()
            apply_privacy(model, privacy_mode, n_samples=len(X_b))
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            optimizer.step()
        tau       = n_steps
        local_st  = model.state_dict()
        norm_grad = {}
        for key in local_st:
            if local_st[key].dtype == torch.float32:
                norm_grad[key] = (local_st[key] - global_state[key].to(dev)) / tau
            else:
                norm_grad[key] = local_st[key].clone()
        del global_state
        aux = {"norm_grad": norm_grad, "tau": tau}

    else:
        c_l        = [cv.to(dev) for cv in c_local]  if c_local  else None
        c_g        = [cv.to(dev) for cv in c_global] if c_global else None
        step_count = 0

        for _ in range(epochs):
            for X_b, y_b in loader:
                X_b, y_b = X_b.to(dev), y_b.to(dev)
                optimizer.zero_grad()
                pred = model(X_b)
                loss = criterion(pred, y_b)
                if global_model is not None and mu > 0:
                    prox = sum(
                        ((p - pg.detach())**2).sum()
                        for p, pg in zip(model.parameters(), global_model.parameters())
                    )
                    loss = loss + (mu / 2) * prox
                loss.backward()
                if c_l is not None and c_g is not None:
                    for param, cl, cg in zip(model.parameters(), c_l, c_g):
                        if param.grad is not None:
                            param.grad.data.add_(cg - cl)
                apply_privacy(model, privacy_mode, n_samples=len(X_b))
                torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
                optimizer.step()
                step_count += 1

        if c_l is not None:
            new_c = [
                cl - cg + torch.zeros_like(cl) / (step_count * lr)
                for cl, cg in zip(c_l, c_g)
            ]
            aux = {"new_c_local": [c.cpu() for c in new_c]}

    return model, aux


def local_train_lr(lr_wrapper, X, y):
    if len(np.unique(y)) < 2:
        return lr_wrapper
    X_r, y_r = smote_bank(X, y)
    if len(np.unique(y_r)) < 2:
        return lr_wrapper
    lr_wrapper.fit(X_r, y_r)
    return lr_wrapper


# ══════════════════════════════════════════════════════════════════
#  METRICS
# ══════════════════════════════════════════════════════════════════

def get_probs_nn(model, X):
    model.eval()
    with torch.no_grad():
        return model(
            torch.tensor(X, dtype=torch.float32).to(DEVICE)
        ).cpu().numpy()


def compute_metrics(y_true, y_prob, k_pct=0.005):
    pred        = (y_prob >= 0.5).astype(int)
    auc         = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else 0.5
    f1          = f1_score(y_true, pred, zero_division=0)
    prec        = precision_score(y_true, pred, zero_division=0)
    rec         = recall_score(y_true, pred, zero_division=0)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    ks          = float(np.max(tpr - fpr))
    idx         = np.where(fpr <= 0.01)[0]
    r1fpr       = float(tpr[idx[-1]]) if len(idx) else 0.0
    K           = max(1, int(len(y_true) * k_pct))
    pk          = float(y_true[np.argsort(y_prob)[::-1][:K]].mean())
    return dict(auc=round(auc,6), f1=round(f1,6), precision=round(prec,6),
                recall=round(rec,6), ks_stat=round(ks,6),
                recall_1fpr=round(r1fpr,6), prec_at_k=round(pk,6))


def fairness_metrics(bank_aucs):
    aucs = [a for a in bank_aucs if a is not None]
    if len(aucs) < 2:
        return 0.0, 1.0
    sigma = float(np.std(aucs))
    jfi   = sum(aucs)**2 / (len(aucs) * sum(a**2 for a in aucs))
    return round(sigma, 6), round(jfi, 6)


# ══════════════════════════════════════════════════════════════════
#  CHECKPOINT & LOGGING
# ══════════════════════════════════════════════════════════════════

def combo_dir(fl, ml, priv, output_root=OUTPUT_ROOT):
    name = f"{fl}_{ml}_{priv}"
    path = os.path.join(output_root, name)
    os.makedirs(path, exist_ok=True)
    return path, name


def save_checkpoint(path, name, rnd, model, metrics, is_lr=False):
    if rnd % HP["SAVE_CKPT_EVERY"] != 0:
        return
    fname = os.path.join(path, f"checkpoint_{name}_round{rnd:03d}.pt")
    if is_lr:
        torch.save({"round": rnd, "metrics": metrics,
                    "model_params": model.get_params()}, fname)
    else:
        torch.save({"round": rnd, "metrics": metrics,
                    "model_state": model.state_dict()}, fname)


def find_latest_checkpoint(path, name):
    ckpts = sorted([
        f for f in os.listdir(path)
        if f.startswith(f"checkpoint_{name}_round") and f.endswith(".pt")
    ])
    if not ckpts:
        return 0, None
    latest = os.path.join(path, ckpts[-1])
    ckpt   = torch.load(latest, map_location="cpu", weights_only=False)
    rnd    = ckpt["round"]
    print(f"  Resuming from checkpoint: round {rnd}")
    return rnd, ckpt


def append_csv_row(csv_path, row_dict, write_header=False):
    pd.DataFrame([row_dict]).to_csv(
        csv_path, mode="a",
        header=write_header or not os.path.exists(csv_path),
        index=False
    )


def save_summary(path, name, best_round, best_auc, best_f1, final_metrics, bank_aucs_final):
    summary = {
        "combination"      : name,
        "best_round"       : best_round,
        "best_auc"         : best_auc,
        "best_f1"          : best_f1,
        "final_metrics"    : final_metrics,
        "bank_aucs_final"  : bank_aucs_final,
    }
    with open(os.path.join(path, f"summary_{name}.json"), "w") as f:
        json.dump(summary, f, indent=2)


def append_master(row, master_csv):
    write_header = not os.path.exists(master_csv)
    pd.DataFrame([row]).to_csv(master_csv, mode="a", header=write_header, index=False)

GPU : Tesla P100-PCIE-16GB
VRAM: 17.1 GB
Device : cuda
Output : outputs/


In [2]:
"""
FL Benchmarking — FL ALGORITHMS
Shared across all model notebooks.
Run notebook_base.py first to define all dependencies.
"""

# ══════════════════════════════════════════════════════════════════
#  AGGREGATION
# ══════════════════════════════════════════════════════════════════

def fedavg_aggregate(local_weights, local_sizes):
    total     = sum(local_sizes)
    new_state = copy.deepcopy(local_weights[0])
    for key in new_state:
        new_state[key] = sum(
            local_weights[i][key] * (local_sizes[i] / total)
            for i in range(len(local_sizes))
        )
    return new_state


# ══════════════════════════════════════════════════════════════════
#  FEDAVG / FEDPROX / PERSFL
# ══════════════════════════════════════════════════════════════════

def run_fedavg(global_model, bank_data, X_te, y_te,
               fl_algo, privacy_mode, path, name, master_csv,
               start_round=1):
    is_lr    = isinstance(global_model, LRWrapper)
    is_perfl = (fl_algo == "PersFL")
    mu       = HP["FEDPROX_MU"] if fl_algo == "FedProx" else 0.0
    n_banks  = len(bank_data)

    # PersFL: save personal models to disk instead of keeping in memory
    personal_model_paths = {}
    if is_perfl and not is_lr:
        for i in range(n_banks):
            pm      = copy.deepcopy(global_model)
            pm_path = os.path.join(path, f"personal_bank{i}.pt")
            torch.save(pm.state_dict(), pm_path)
            personal_model_paths[i] = pm_path
            del pm
        torch.cuda.empty_cache()

    csv_path    = os.path.join(path, f"results_{name}.csv")
    best_auc, best_f1, best_round = 0.0, 0.0, 1
    rounds_95   = None
    all_metrics = []
    bank_aucs   = []

    for rnd in range(start_round, HP["FL_ROUNDS"] + 1):
        t0            = time.time()
        local_weights = []
        local_sizes   = []

        for bank_id, (X_b, y_b) in enumerate(bank_data):
            X_r, y_r = smote_bank(X_b, y_b)
            if is_lr:
                lm = copy.deepcopy(global_model)
                local_train_lr(lm, X_r, y_r)
                local_weights.append(lm.get_params())
                local_sizes.append(len(X_b))
                del lm
            else:
                lm     = copy.deepcopy(global_model)
                loader = make_loader(X_r, y_r)
                gm_ref = copy.deepcopy(global_model) if mu > 0 else None
                lm, _  = local_train_nn(
                    lm, loader, HP["LOCAL_EPOCHS"], HP["LR"],
                    privacy_mode, global_model=gm_ref, mu=mu
                )
                local_weights.append(copy.deepcopy(lm.state_dict()))
                local_sizes.append(len(X_b))
                del lm, loader
                if gm_ref is not None:
                    del gm_ref
                torch.cuda.empty_cache()

        # Aggregate
        if is_lr:
            total = sum(local_sizes)
            global_model.set_params({
                "coef": sum(
                    local_weights[i]["coef"] * (local_sizes[i] / total)
                    for i in range(n_banks)
                ),
                "intercept": sum(
                    local_weights[i]["intercept"] * (local_sizes[i] / total)
                    for i in range(n_banks)
                )
            })
        else:
            new_state = fedavg_aggregate(local_weights, local_sizes)
            global_model.load_state_dict(new_state)
            del new_state

        del local_weights
        torch.cuda.empty_cache()

        # PersFL fine-tuning — one model at a time, save to disk
        if is_perfl and not is_lr:
            for bank_id, (X_b, y_b) in enumerate(bank_data):
                X_r, y_r = smote_bank(X_b, y_b)
                pm       = copy.deepcopy(global_model)
                loader   = make_loader(X_r, y_r)
                pm, _    = local_train_nn(
                    pm, loader, HP["FINETUNE_EPOCHS"], HP["FINETUNE_LR"], "NoDP"
                )
                torch.save(pm.state_dict(), personal_model_paths[bank_id])
                del pm, loader
                torch.cuda.empty_cache()

        # Evaluate
        probs = global_model.predict_proba(X_te) if is_lr else get_probs_nn(global_model, X_te)
        m     = compute_metrics(y_te, probs)
        loss  = float(-(y_te * np.log(probs + 1e-9) +
                        (1 - y_te) * np.log(1 - probs + 1e-9)).mean())

        # Per-bank fairness
        bank_aucs = []
        for i, (X_b, y_b) in enumerate(bank_data):
            if len(np.unique(y_b)) < 2:
                bank_aucs.append(None)
                continue
            if is_perfl and not is_lr and i in personal_model_paths:
                eval_m = copy.deepcopy(global_model)
                eval_m.load_state_dict(
                    torch.load(personal_model_paths[i], map_location=DEVICE, weights_only=False)
                )
                p = get_probs_nn(eval_m, X_b)
                del eval_m
                torch.cuda.empty_cache()
            elif is_lr:
                p = global_model.predict_proba(X_b)
            else:
                p = get_probs_nn(global_model, X_b)
            bank_aucs.append(round(roc_auc_score(y_b, p), 6))

        sigma, jfi = fairness_metrics(bank_aucs)

        if m["auc"] > best_auc:
            best_auc, best_f1, best_round = m["auc"], m["f1"], rnd
        if rounds_95 is None and m["auc"] >= 0.95 * best_auc:
            rounds_95 = rnd

        row = {"round": rnd, **m, "sigma_auc": sigma, "jfi": jfi,
               "loss": round(loss, 6), "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
        append_csv_row(csv_path, row, write_header=(rnd == start_round))
        all_metrics.append(row)
        save_checkpoint(path, name, rnd, global_model, m, is_lr=is_lr)

        print(f"  Round {rnd:03d}/{HP['FL_ROUNDS']:03d} | "
              f"AUC: {m['auc']:.4f} | F1: {m['f1']:.4f} | "
              f"R@1%FPR: {m['recall_1fpr']:.4f} | P@K: {m['prec_at_k']:.4f} | "
              f"KS: {m['ks_stat']:.4f} | σ: {sigma:.4f} | Loss: {loss:.4f}")

    return global_model, all_metrics, best_auc, best_f1, best_round, rounds_95, bank_aucs


# ══════════════════════════════════════════════════════════════════
#  SCAFFOLD
# ══════════════════════════════════════════════════════════════════

def run_scaffold(global_model, bank_data, X_te, y_te,
                 privacy_mode, path, name, master_csv,
                 start_round=1):
    n_banks  = len(bank_data)
    c_global = [torch.zeros_like(p.data).cpu() for p in global_model.parameters()]
    c_locals = [[torch.zeros_like(p.data).cpu() for p in global_model.parameters()]
                for _ in range(n_banks)]

    csv_path    = os.path.join(path, f"results_{name}.csv")
    best_auc, best_f1, best_round = 0.0, 0.0, 1
    rounds_95   = None
    all_metrics = []
    bank_aucs   = []

    for rnd in range(start_round, HP["FL_ROUNDS"] + 1):
        t0            = time.time()
        local_weights = []
        local_sizes   = []
        delta_c_list  = []

        for bank_id, (X_b, y_b) in enumerate(bank_data):
            X_r, y_r = smote_bank(X_b, y_b)
            lm       = copy.deepcopy(global_model)
            loader   = make_loader(X_r, y_r)
            lm, aux  = local_train_nn(
                lm, loader, HP["LOCAL_EPOCHS"], HP["LR"], privacy_mode,
                c_local=c_locals[bank_id], c_global=c_global
            )
            if "new_c_local" in aux:
                delta = [nc - oc for nc, oc in zip(aux["new_c_local"], c_locals[bank_id])]
                c_locals[bank_id] = aux["new_c_local"]
                delta_c_list.append(delta)
            local_weights.append(copy.deepcopy(lm.state_dict()))
            local_sizes.append(len(X_b))
            del lm, loader, aux
            torch.cuda.empty_cache()

        new_state = fedavg_aggregate(local_weights, local_sizes)
        global_model.load_state_dict(new_state)
        del new_state, local_weights
        torch.cuda.empty_cache()

        if delta_c_list:
            for j in range(len(c_global)):
                c_global[j] = c_global[j] + sum(
                    delta_c_list[i][j] for i in range(len(delta_c_list))
                ) / n_banks
        del delta_c_list

        probs = get_probs_nn(global_model, X_te)
        m     = compute_metrics(y_te, probs)
        loss  = float(-(y_te * np.log(probs + 1e-9) +
                        (1 - y_te) * np.log(1 - probs + 1e-9)).mean())

        bank_aucs = []
        for i, (X_b, y_b) in enumerate(bank_data):
            if len(np.unique(y_b)) < 2:
                bank_aucs.append(None)
                continue
            p = get_probs_nn(global_model, X_b)
            bank_aucs.append(round(roc_auc_score(y_b, p), 6))

        sigma, jfi = fairness_metrics(bank_aucs)

        if m["auc"] > best_auc:
            best_auc, best_f1, best_round = m["auc"], m["f1"], rnd
        if rounds_95 is None and m["auc"] >= 0.95 * best_auc:
            rounds_95 = rnd

        row = {"round": rnd, **m, "sigma_auc": sigma, "jfi": jfi,
               "loss": round(loss, 6), "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
        append_csv_row(csv_path, row, write_header=(rnd == start_round))
        all_metrics.append(row)
        save_checkpoint(path, name, rnd, global_model, m)

        print(f"  Round {rnd:03d}/{HP['FL_ROUNDS']:03d} | "
              f"AUC: {m['auc']:.4f} | F1: {m['f1']:.4f} | "
              f"R@1%FPR: {m['recall_1fpr']:.4f} | P@K: {m['prec_at_k']:.4f} | "
              f"KS: {m['ks_stat']:.4f} | σ: {sigma:.4f} | Loss: {loss:.4f}")

    return global_model, all_metrics, best_auc, best_f1, best_round, rounds_95, bank_aucs


# ══════════════════════════════════════════════════════════════════
#  FEDNOVA
# ══════════════════════════════════════════════════════════════════

def run_fednova(global_model, bank_data, X_te, y_te,
                privacy_mode, path, name, master_csv,
                start_round=1):
    n_banks     = len(bank_data)
    csv_path    = os.path.join(path, f"results_{name}.csv")
    best_auc, best_f1, best_round = 0.0, 0.0, 1
    rounds_95   = None
    all_metrics = []
    bank_aucs   = []

    for rnd in range(start_round, HP["FL_ROUNDS"] + 1):
        t0           = time.time()
        global_state = copy.deepcopy(global_model.state_dict())
        norm_grads   = []
        tau_list     = []
        local_sizes  = []

        for bank_id, (X_b, y_b) in enumerate(bank_data):
            X_r, y_r = smote_bank(X_b, y_b)
            lm       = copy.deepcopy(global_model)
            loader   = make_loader(X_r, y_r)
            lm, aux  = local_train_nn(
                lm, loader, None, HP["LR"], privacy_mode,
                use_steps=True, n_steps=HP["LOCAL_STEPS"]
            )
            norm_grads.append(aux["norm_grad"])
            tau_list.append(aux["tau"])
            local_sizes.append(len(X_b))
            del lm, loader, aux
            torch.cuda.empty_cache()

        total_w   = sum(local_sizes[i] * tau_list[i] for i in range(n_banks))
        new_state = copy.deepcopy(global_state)
        for key in new_state:
            if new_state[key].dtype == torch.float32:
                agg = sum(
                    norm_grads[i][key].to(DEVICE) *
                    (local_sizes[i] * tau_list[i] / total_w)
                    for i in range(n_banks)
                )
                new_state[key] = global_state[key].to(DEVICE) + agg
            else:
                new_state[key] = sum(
                    norm_grads[i][key].to(DEVICE) * (local_sizes[i] / sum(local_sizes))
                    for i in range(n_banks)
                )
        global_model.load_state_dict(new_state)
        del norm_grads, new_state, global_state
        torch.cuda.empty_cache()

        probs = get_probs_nn(global_model, X_te)
        m     = compute_metrics(y_te, probs)
        loss  = float(-(y_te * np.log(probs + 1e-9) +
                        (1 - y_te) * np.log(1 - probs + 1e-9)).mean())

        bank_aucs = []
        for i, (X_b, y_b) in enumerate(bank_data):
            if len(np.unique(y_b)) < 2:
                bank_aucs.append(None)
                continue
            p = get_probs_nn(global_model, X_b)
            bank_aucs.append(round(roc_auc_score(y_b, p), 6))

        sigma, jfi = fairness_metrics(bank_aucs)

        if m["auc"] > best_auc:
            best_auc, best_f1, best_round = m["auc"], m["f1"], rnd
        if rounds_95 is None and m["auc"] >= 0.95 * best_auc:
            rounds_95 = rnd

        row = {"round": rnd, **m, "sigma_auc": sigma, "jfi": jfi,
               "loss": round(loss, 6), "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
        append_csv_row(csv_path, row, write_header=(rnd == start_round))
        all_metrics.append(row)
        save_checkpoint(path, name, rnd, global_model, m)

        print(f"  Round {rnd:03d}/{HP['FL_ROUNDS']:03d} | "
              f"AUC: {m['auc']:.4f} | F1: {m['f1']:.4f} | "
              f"R@1%FPR: {m['recall_1fpr']:.4f} | P@K: {m['prec_at_k']:.4f} | "
              f"KS: {m['ks_stat']:.4f} | σ: {sigma:.4f} | Loss: {loss:.4f}")

    return global_model, all_metrics, best_auc, best_f1, best_round, rounds_95, bank_aucs


# ══════════════════════════════════════════════════════════════════
#  RUN ONE COMBINATION
# ══════════════════════════════════════════════════════════════════

def run_combination(fl_algo, ml_model, privacy_mode,
                    X_tr, X_te, y_tr, y_te,
                    combo_idx, total_combos, master_csv):
    gc.collect()
    torch.cuda.empty_cache()

    path, name = combo_dir(fl_algo, ml_model, privacy_mode)

    print(f"\n{'='*60}")
    print(f"  {fl_algo} + {ml_model} + {privacy_mode}  "
          f"({combo_idx}/{total_combos})")
    print(f"{'='*60}")

    start_round, ckpt = find_latest_checkpoint(path, name)
    start_round += 1

    n_features   = X_tr.shape[1]
    bank_data    = non_iid_split(X_tr, y_tr)
    global_model = build_model(ml_model, n_features)

    if ckpt is not None:
        if isinstance(global_model, LRWrapper):
            global_model.set_params(ckpt["model_params"])
        else:
            global_model.load_state_dict(
                {k: v.to(DEVICE) for k, v in ckpt["model_state"].items()}
            )

    if isinstance(global_model, LRWrapper) and fl_algo in ["SCAFFOLD", "FedNova"]:
        print(f"  SKIP: LR not compatible with {fl_algo}")
        return None

    t_start = time.time()

    if fl_algo in ["FedAvg", "FedProx", "PersFL"]:
        _, all_m, best_auc, best_f1, best_round, rounds_95, bank_aucs = run_fedavg(
            global_model, bank_data, X_te, y_te,
            fl_algo, privacy_mode, path, name, master_csv,
            start_round=start_round
        )
    elif fl_algo == "SCAFFOLD":
        _, all_m, best_auc, best_f1, best_round, rounds_95, bank_aucs = run_scaffold(
            global_model, bank_data, X_te, y_te,
            privacy_mode, path, name, master_csv,
            start_round=start_round
        )
    elif fl_algo == "FedNova":
        if isinstance(global_model, LRWrapper):
            print("  SKIP: LR not compatible with FedNova")
            return None
        _, all_m, best_auc, best_f1, best_round, rounds_95, bank_aucs = run_fednova(
            global_model, bank_data, X_te, y_te,
            privacy_mode, path, name, master_csv,
            start_round=start_round
        )

    total_time = round(time.time() - t_start, 1)
    final_m    = all_m[-1] if all_m else {}
    sigma, jfi = fairness_metrics(bank_aucs)

    save_summary(path, name, best_round, best_auc, best_f1, final_m, bank_aucs)

    master_row = {
        "fl_algorithm"      : fl_algo,
        "ml_model"          : ml_model,
        "privacy_mode"      : privacy_mode,
        "best_auc"          : best_auc,
        "best_auc_round"    : best_round,
        "final_auc"         : final_m.get("auc", 0),
        "best_f1"           : best_f1,
        "best_recall_1fpr"  : max((r.get("recall_1fpr", 0) for r in all_m), default=0),
        "ks_stat"           : final_m.get("ks_stat", 0),
        "sigma_auc"         : sigma,
        "jfi"               : jfi,
        "rounds_to_95pct"   : rounds_95,
        "total_time_seconds": total_time,
    }
    append_master(master_row, master_csv)

    # Cleanup
    del global_model, bank_data
    gc.collect()
    torch.cuda.empty_cache()

    print(f"\n  ✓ Done | Best AUC: {best_auc:.4f} @ round {best_round} | Time: {total_time}s")
    return master_row


def run_combination_safe(fl_algo, ml_model, privacy_mode,
                         X_tr, X_te, y_tr, y_te,
                         combo_idx, total_combos, master_csv):
    gc.collect()
    torch.cuda.empty_cache()
    try:
        result = run_combination(
            fl_algo, ml_model, privacy_mode,
            X_tr, X_te, y_tr, y_te,
            combo_idx, total_combos, master_csv
        )
    except Exception as e:
        print(f"  ✗ FAILED: {fl_algo} + {ml_model} + {privacy_mode} | Error: {e}")
        result = None
    finally:
        gc.collect()
        torch.cuda.empty_cache()
    return result

In [3]:
"""
═══════════════════════════════════════════════════════════════════
FL Benchmark — NOTEBOOK: LR
Runs: LR × 5 FL algos × 3 privacy = 15 combinations × 60 rounds
Note: LR skips SCAFFOLD and FedNova (incompatible) = 9 actual runs
Paste notebook_base.py and notebook_fl_algorithms.py above this cell
═══════════════════════════════════════════════════════════════════
"""

MODEL     = "LR"
MASTER_CSV = os.path.join(OUTPUT_ROOT, f"master_{MODEL}.csv")

DATA_PATH = "/kaggle/input/datasets/organizations/mlg-ulb/creditcardfraud/creditcard.csv"

# ── Run ───────────────────────────────────────────────────────────
X_tr, X_te, y_tr, y_te = load_data(DATA_PATH)

combos = [(fl, MODEL, pr) for fl in ALL_FL for pr in ALL_PRIVACY]
total  = len(combos)

print(f"\nRunning {total} combinations for model: {MODEL}")
print(f"Results → {MASTER_CSV}\n")

results = []
for idx, (fl, ml, pr) in enumerate(combos, 1):
    r = run_combination_safe(fl, ml, pr, X_tr, X_te, y_tr, y_te,
                             idx, total, MASTER_CSV)
    if r:
        results.append(r)

print(f"\n{'='*60}")
print(f"DONE — {MODEL} | {len(results)}/{total} completed")
print(f"{'='*60}")
print(pd.read_csv(MASTER_CSV).to_string(index=False))


Loading dataset...
  Total records : 284,807
  Fraud rate    : 0.17%

Running 15 combinations for model: LR
Results → outputs/master_LR.csv


  FedAvg + LR + NoDP  (1/15)

Non-IID split (Dirichlet α=0.4):
  Bank 1: 45,599 records | fraud 0.24%
  Bank 2: 45,672 records | fraud 0.40%
  Bank 3: 45,490 records | fraud 0.00%
  Bank 4: 45,592 records | fraud 0.22%
  Bank 5: 45,491 records | fraud 0.00%
  Round 001/060 | AUC: 0.9524 | F1: 0.5631 | R@1%FPR: 0.8980 | P@K: 0.3063 | KS: 0.8923 | σ: 0.0102 | Loss: 0.0237
  Round 002/060 | AUC: 0.9588 | F1: 0.5939 | R@1%FPR: 0.8980 | P@K: 0.3063 | KS: 0.8923 | σ: 0.0103 | Loss: 0.0169
  Round 003/060 | AUC: 0.9573 | F1: 0.5541 | R@1%FPR: 0.8980 | P@K: 0.3063 | KS: 0.8908 | σ: 0.0105 | Loss: 0.0170
  Round 004/060 | AUC: 0.9562 | F1: 0.6667 | R@1%FPR: 0.8980 | P@K: 0.3063 | KS: 0.8945 | σ: 0.0103 | Loss: 0.0159
  Round 005/060 | AUC: 0.9585 | F1: 0.5088 | R@1%FPR: 0.8980 | P@K: 0.3063 | KS: 0.8932 | σ: 0.0095 | Loss: 0.0205
  Round 006/060 | AUC: 0